# Deploy the AI pipeline

Deploys the contract-renewal AI pipeline into an AWS account: **EventBridge (daily) → Lambda 1 → Bedrock batch inference → EventBridge (on complete) → Lambda 2 → Aurora PostgreSQL**, with prompts in **Bedrock Prompt Management** and contract data read from **Snowflake**.

What it creates (all through one CloudFormation stack, so it is repeatable and reviewable in `template.yaml`):

| Piece | Purpose |
|---|---|
| S3 bucket | `PCR/AI_Input/<job>/<run>/` and `PCR/AI_Output/<job>/<run>/` for batch files (kept 90 days) |
| Lambda 1 `start-batches` | decides what needs the model, builds the batch inputs, starts the Bedrock jobs |
| Lambda 2 `on-batch-complete` | handles each finished job: writes summaries, or chains a recommendation through its **evaluation** and **draft** passes; also runs the scheduled **sweep** |
| EventBridge rules | daily start · Bedrock "job finished" events · 15-minute sweep (catches any missed event) |
| IAM roles | one for the Lambdas, one for Bedrock to read/write the batch files |
| Alarms (optional) | notify an SNS topic when either Lambda errors |

**Run the cells in order.** Each one is safe to run again. Cells that change your account say so.

### Before you start

1. **AWS permissions** to create IAM roles, Lambda functions, S3 buckets, EventBridge rules, Secrets Manager secrets, CloudFormation stacks and Bedrock prompts.
2. **Bedrock model access** granted in your region, for a model that supports *batch inference* there. (Not every model/region does - check the Bedrock console. Batch jobs also need at least 100 records; smaller runs are done directly, automatically.)
3. **Aurora PostgreSQL** with the application's migrations `0001`-`0005` applied and `python -m ingest` run at least once (that creates the work-list views the pipeline reads).
4. **A Snowflake user** that can `SELECT` the contract and claims tables (key-pair authentication recommended).
5. **Network:** private subnets that can reach Aurora, plus a NAT gateway (or VPC endpoints for S3, Secrets Manager, Bedrock runtime, Bedrock agent, Lambda and CloudWatch Logs) and outbound HTTPS to Snowflake.
6. **On this machine:** Python 3.12+, `pip install boto3`, internet access for `pip`.

## 1 · Settings
Edit this one cell. Nothing else needs changing.

In [ ]:
CONFIG = {
    "profile": None,                         # an AWS CLI profile name, or None for your default credentials
    "region": "eu-west-1",
    "prefix": "pcr-ai",                      # names every resource; use another prefix for a second environment

    "artifact_bucket": "",                   # bucket for the packaged code (created if it does not exist)
    "pipeline_bucket": "",                   # NEW bucket for batch input/output - must be globally unique

    "subnet_ids": [],                        # e.g. ["subnet-0abc...", "subnet-0def..."]  (private, can reach Aurora)
    "security_group_ids": [],                # e.g. ["sg-0abc..."]                         (egress 443 + Aurora's port)

    "pg_secret_name": "pcr-ai/postgres-ai-writer",
    "snowflake_secret_name": "pcr-ai/snowflake",
    "snowflake_contract_table": "",          # e.g. "DADP_DB.DADP_CST.<contract data product>"
    "snowflake_claims_table": "",            # e.g. "DADP_DB.DADP_CST.<web claims>"

    "model_id": "",                          # a Bedrock model / inference-profile id that supports batch inference
    "db_schema": "ai_recommendations",
    "daily_schedule": "cron(0 3 * * ? *)",   # UTC. The previous evening's data load should be finished by then.
    "alarm_topic_arn": "",                   # optional SNS topic for error alarms
}

STACK_NAME = f"{CONFIG['prefix']}-pipeline"

In [ ]:
import json
import sys
import time
import getpass
from pathlib import Path

import boto3
from botocore.config import Config
from botocore.exceptions import ClientError

assert sys.version_info >= (3, 9), "use a recent Python"
HERE = Path.cwd()
for needed in ("template.yaml", "package.py", "prompts", "pipeline", "lambdas"):
    assert (HERE / needed).exists(), f"run this notebook from the aws/ folder (missing {needed})"

missing = [k for k in ("artifact_bucket", "pipeline_bucket", "subnet_ids", "security_group_ids", "snowflake_contract_table",
                       "snowflake_claims_table", "model_id") if not CONFIG[k]]
assert not missing, f"fill in CONFIG first: {missing}"

session = boto3.Session(profile_name=CONFIG["profile"], region_name=CONFIG["region"])
identity = session.client("sts").get_caller_identity()
ACCOUNT = identity["Account"]
print(f"Account {ACCOUNT} · region {CONFIG['region']} · as {identity['Arn']}")

STATE_FILE = HERE / "deploy_state.json"          # remembers prompt ids, secret ARNs and the last build between cells and sessions
STATE = json.loads(STATE_FILE.read_text()) if STATE_FILE.exists() else {}


def save_state():
    STATE_FILE.write_text(json.dumps(STATE, indent=2))

## 2 · Secrets  *(creates / updates two secrets in Secrets Manager)*

* **Postgres** - the connection for the restricted `ai_writer` database role (not an admin). The role is created in step 6.
* **Snowflake** - a key-pair user is preferred; a password also works.

Values are typed here at the prompt and never written to the notebook or to disk. Answer `n` to keep an existing secret as it is.

In [ ]:
sm = session.client("secretsmanager")


def put_secret(name, payload, description):
    try:
        return sm.create_secret(Name=name, Description=description, SecretString=json.dumps(payload))["ARN"]
    except sm.exceptions.ResourceExistsException:
        return sm.put_secret_value(SecretId=name, SecretString=json.dumps(payload))["ARN"]


def secret_arn(name):
    try:
        return sm.describe_secret(SecretId=name)["ARN"]
    except sm.exceptions.ResourceNotFoundException:
        return None


def ask(prompt, default=""):
    value = input(f"{prompt}{f' [{default}]' if default else ''}: ").strip()
    return value or default


for key, name in (("pg_secret_arn", CONFIG["pg_secret_name"]), ("snowflake_secret_arn", CONFIG["snowflake_secret_name"])):
    existing = secret_arn(name)
    if existing and ask(f"Secret {name} exists - replace its value? (y/N)", "n").lower() != "y":
        STATE[key] = existing
        continue
    if key == "pg_secret_arn":
        payload = {"host": ask("Aurora writer endpoint"), "port": int(ask("Port", "5432")), "dbname": ask("Database", "uda_1325"),
                   "username": ask("Role", "ai_writer"), "password": getpass.getpass("Password for that role: ")}
        STATE[key] = put_secret(name, payload, "Aurora connection for the AI pipeline's restricted role")
    else:
        payload = {"account": ask("Snowflake account identifier"), "user": ask("User"), "warehouse": ask("Warehouse"),
                   "role": ask("Role (optional)"), "database": ask("Database (optional)"), "schema": ask("Schema (optional)")}
        pem = ask("Path to the private key PEM file (leave empty to use a password)")
        if pem:
            payload["private_key"] = Path(pem).expanduser().read_text()
            passphrase = getpass.getpass("Key passphrase (empty if none): ")
            if passphrase:
                payload["private_key_passphrase"] = passphrase
        else:
            payload["password"] = getpass.getpass("Snowflake password: ")
        STATE[key] = put_secret(name, {k: v for k, v in payload.items() if v}, "Snowflake connection for the AI pipeline (read only)")
    print("stored", name)
save_state()
print({k: v for k, v in STATE.items() if k.endswith("_arn")})

## 3 · Prompts  *(creates / updates six prompts in Bedrock Prompt Management)*

The wording of every prompt lives in `prompts/*.json`. This cell publishes each as a **new immutable version**, and the pipeline is pointed at exactly that version - so changing a prompt later is: edit in the Prompt Management console (or the JSON), create a version, and update the stack's `PromptsJson`. Nothing is redeployed.

In [ ]:
agent = session.client("bedrock-agent")
JOBS = ["contract_summary", "web_claims_summary", "account_summary", "recommendation", "evaluation", "draft"]


def find_prompt_id(name):
    token = None
    while True:
        resp = agent.list_prompts(maxResults=100, **({"nextToken": token} if token else {}))
        for p in resp.get("promptSummaries", []):
            if p["name"] == name:
                return p["id"]
        token = resp.get("nextToken")
        if not token:
            return None


def publish_prompt(job):
    spec = json.loads((HERE / "prompts" / f"{job}.json").read_text())
    variant = {
        "name": "default", "templateType": "TEXT", "modelId": CONFIG["model_id"],
        "inferenceConfiguration": {"text": {"maxTokens": spec["inference"]["maxTokens"], "temperature": spec["inference"]["temperature"]}},
        "templateConfiguration": {"text": {"text": spec["template"], "inputVariables": [{"name": v} for v in spec["variables"]]}},
    }
    pid = find_prompt_id(spec["name"])
    if pid:
        agent.update_prompt(promptIdentifier=pid, name=spec["name"], description=spec["description"], defaultVariant="default", variants=[variant])
    else:
        pid = agent.create_prompt(name=spec["name"], description=spec["description"], defaultVariant="default", variants=[variant])["id"]
    version = agent.create_prompt_version(promptIdentifier=pid, description="published from the deployment notebook")["version"]
    return {"id": pid, "version": str(version)}


STATE["prompts"] = {job: publish_prompt(job) for job in JOBS}
save_state()
for job, ref in STATE["prompts"].items():
    print(f"{job:20s} {ref['id']}  v{ref['version']}")

## 4 · Package and upload

Builds the dependency layer (psycopg, snowflake-connector) from pre-built Linux wheels and zips the code. **No Docker needed, and it works from Windows or macOS.** Expect about 15 MB for the layer.

In [ ]:
import package

BUILD = HERE / "build"
BUILD.mkdir(exist_ok=True)
code_zip, code_size = package.build_code(BUILD)
layer_zip, layer_size = package.build_layer(BUILD)
print(f"code {code_zip.stat().st_size / 1e6:.1f} MB · layer {layer_zip.stat().st_size / 1e6:.1f} MB zipped "
      f"({(code_size + layer_size) / 1e6:.0f} MB unzipped of Lambda's 250 MB)")

s3 = session.client("s3")
try:
    s3.head_bucket(Bucket=CONFIG["artifact_bucket"])
except ClientError:
    args = {"Bucket": CONFIG["artifact_bucket"]}
    if CONFIG["region"] != "us-east-1":
        args["CreateBucketConfiguration"] = {"LocationConstraint": CONFIG["region"]}
    s3.create_bucket(**args)
    s3.put_public_access_block(Bucket=CONFIG["artifact_bucket"], PublicAccessBlockConfiguration={
        "BlockPublicAcls": True, "IgnorePublicAcls": True, "BlockPublicPolicy": True, "RestrictPublicBuckets": True})
    print("created artifact bucket", CONFIG["artifact_bucket"])

stamp = time.strftime("%Y%m%d-%H%M%S")
STATE["code_key"], STATE["layer_key"] = f"{CONFIG['prefix']}/{stamp}/pipeline-code.zip", f"{CONFIG['prefix']}/{stamp}/dependencies-layer.zip"
s3.upload_file(str(code_zip), CONFIG["artifact_bucket"], STATE["code_key"])
s3.upload_file(str(layer_zip), CONFIG["artifact_bucket"], STATE["layer_key"])
save_state()
print("uploaded to s3://%s/%s" % (CONFIG["artifact_bucket"], f"{CONFIG['prefix']}/{stamp}/"))

## 5 · Deploy the stack  *(creates / updates the CloudFormation stack)*

Creates the bucket, roles, layer, both Lambdas and the EventBridge rules. Running it again after a code or prompt change **updates** the stack in place.

In [ ]:
cfn = session.client("cloudformation")
params = {
    "Prefix": CONFIG["prefix"], "ArtifactBucket": CONFIG["artifact_bucket"], "CodeKey": STATE["code_key"], "LayerKey": STATE["layer_key"],
    "PipelineBucketName": CONFIG["pipeline_bucket"], "PgSecretArn": STATE["pg_secret_arn"], "SnowflakeSecretArn": STATE["snowflake_secret_arn"],
    "SubnetIds": ",".join(CONFIG["subnet_ids"]), "SecurityGroupIds": ",".join(CONFIG["security_group_ids"]),
    "SnowflakeContractTable": CONFIG["snowflake_contract_table"], "SnowflakeClaimsTable": CONFIG["snowflake_claims_table"],
    "ModelId": CONFIG["model_id"], "PromptsJson": json.dumps(STATE["prompts"], separators=(",", ":")), "DbSchema": CONFIG["db_schema"],
    "DailySchedule": CONFIG["daily_schedule"], "AlarmTopicArn": CONFIG["alarm_topic_arn"],
}
template_body = (HERE / "template.yaml").read_text()
cfn.validate_template(TemplateBody=template_body)
kwargs = dict(StackName=STACK_NAME, TemplateBody=template_body, Capabilities=["CAPABILITY_NAMED_IAM"],
              Parameters=[{"ParameterKey": k, "ParameterValue": v} for k, v in params.items()])


def stack_exists():
    try:
        cfn.describe_stacks(StackName=STACK_NAME)
        return True
    except ClientError as e:
        if "does not exist" in str(e):
            return False
        raise


try:
    if stack_exists():
        cfn.update_stack(**kwargs)
        waiter = cfn.get_waiter("stack_update_complete")
        print("updating", STACK_NAME, "...")
    else:
        cfn.create_stack(**kwargs)
        waiter = cfn.get_waiter("stack_create_complete")
        print("creating", STACK_NAME, "... (a few minutes)")
    waiter.wait(StackName=STACK_NAME, WaiterConfig={"Delay": 10, "MaxAttempts": 90})
except ClientError as e:
    if "No updates are to be performed" in str(e):
        print("the stack is already up to date")
    else:
        raise
except Exception:
    for ev in cfn.describe_stack_events(StackName=STACK_NAME)["StackEvents"][:15]:
        if "FAILED" in ev["ResourceStatus"]:
            print(ev["LogicalResourceId"], "-", ev.get("ResourceStatusReason"))
    raise

OUTPUTS = {o["OutputKey"]: o["OutputValue"] for o in cfn.describe_stacks(StackName=STACK_NAME)["Stacks"][0].get("Outputs", [])}
START, COMPLETE = OUTPUTS["StartFunctionName"], OUTPUTS["CompleteFunctionName"]
print(json.dumps(OUTPUTS, indent=2))

## 6 · Database role *(run this SQL in Aurora, once)*

The pipeline connects as a **restricted role** that can add new result versions and drive the batch log, but cannot update or delete anything. Run the statements below as a database administrator (from a bastion, a query editor or `psql`) - Aurora is private, so this notebook cannot do it for you. Use the same password you gave the Postgres secret in step 2.

In [ ]:
S = CONFIG["db_schema"]
print(f"""
CREATE ROLE ai_writer LOGIN PASSWORD '<the password from step 2>';
GRANT USAGE ON SCHEMA {S} TO ai_writer;

-- add new versions of the four results (a trigger numbers the versions); read what exists
GRANT SELECT, INSERT ON {S}.contract_summary, {S}.account_summary, {S}.web_claims_summary, {S}.contract_recommendation TO ai_writer;

-- the batch log is driven only through these functions
GRANT SELECT ON {S}.ai_batch TO ai_writer;
GRANT EXECUTE ON FUNCTION
    {S}.ai_batch_create(uuid, text, uuid, text, jsonb, text, text, integer), {S}.ai_batch_submitted(uuid, text),
    {S}.ai_batch_claim(text, text, interval), {S}.ai_batch_finish(uuid, integer, integer, integer, integer),
    {S}.ai_batch_fail(uuid, text), {S}.ai_batch_fail(text, text)
TO ai_writer;

-- the work lists (the last three exist once `python -m ingest` has run)
GRANT SELECT ON {S}.v_excluded_contracts, {S}.v_retention_options, {S}.v_ai_work, {S}.v_ai_work_accounts TO ai_writer;
""")

## 7 · Smoke test A - connectivity *(starts nothing)*

A **dry run** plans the day's work and counts it: it proves the Lambda can reach Aurora (and read the work lists) and Snowflake (and find the contracts), without starting any job or writing anything.

In [ ]:
lam = session.client("lambda", config=Config(read_timeout=910, retries={"max_attempts": 0}))
logs = session.client("logs")


def invoke(function, payload):
    resp = lam.invoke(FunctionName=function, Payload=json.dumps(payload).encode())
    body = json.loads(resp["Payload"].read())
    if resp.get("FunctionError"):
        raise RuntimeError(json.dumps(body, indent=2)[:3000])
    return body


def tail(function, minutes=10, limit=60):
    start = int((time.time() - minutes * 60) * 1000)
    events = logs.filter_log_events(logGroupName=f"/aws/lambda/{function}", startTime=start, limit=limit).get("events", [])
    for e in events:
        print(e["message"].rstrip()[:400])


result = invoke(START, {"dry_run": True, "limit": 25})
print(json.dumps(result, indent=2))

## 8 · Smoke test B - the whole path on 5 contracts *(starts real, tiny work)*

Runs one job type on 5 contracts. That is below Bedrock's 100-record batch minimum, so it is run directly - which exercises Prompt Management, the model, S3, the batch log and Lambda 2, and writes 5 real summaries. Set `RUN_REAL_SMOKE_TEST = True` to run it.

In [ ]:
RUN_REAL_SMOKE_TEST = False

if RUN_REAL_SMOKE_TEST:
    started = invoke(START, {"limit": 5, "jobs": ["contract_summary"], "force": True})
    print(json.dumps(started, indent=2))
    batch = started["jobs"]["contract_summary"].get("batch")
    for _ in range(36):                                  # up to ~3 minutes
        rows = [r for r in invoke(COMPLETE, {"report": True}) if r["id"] == batch]
        if rows and rows[0]["status"] in ("written", "failed"):
            print(json.dumps(rows[0], indent=2))
            break
        time.sleep(5)
    else:
        print("still running - look at the logs:")
        tail(COMPLETE)
else:
    print("skipped (set RUN_REAL_SMOKE_TEST = True)")

## 9 · The first real batch - what to check

The pieces that only a real Bedrock batch job can prove. After the first scheduled (or manual) run with 100+ records:

1. **The finish event arrives.** Open the CloudWatch logs of the `on-batch-complete` function. The event shape comes from Bedrock; the handler reads `detail.batchJobArn` and `detail.status` (or `batchJobStatus`). If you see "Not a batch completion event", send me the logged event and the field names get adjusted. *Nothing is lost meanwhile - the 15-minute sweep asks Bedrock directly and completes the batch anyway.*
2. **The output location.** Bedrock writes `<output uri>/<job id>/<input file>.jsonl.out`; the handler lists that folder. A `FileNotFoundError` in the logs means the layout differs - the batch is marked failed and nothing is written.
3. **Model and quotas.** A job that fails to start usually means the model does not support batch inference in this region, or a quota. The reason is stored in `ai_batch.error` (admins see it at `GET /api/admin/batches`).

In [ ]:
# Handy operations - run any of these when needed.
def report(n=20):
    """The batch log, newest first."""
    for r in invoke(COMPLETE, {"report": True})[:n]:
        print(f"{r['job_type']:20s} {r['status']:10s} requested={r['records_requested']} written={r['records_written']} "
              f"skipped={r['records_skipped']} failed={r['records_failed']}  {r.get('error') or ''}")


def run_now(**event):
    """Start a run by hand, e.g. run_now(dry_run=True), run_now(jobs=["recommendation"]), run_now(force=True, limit=200)."""
    return invoke(START, event)


def sweep():
    return invoke(COMPLETE, {"sweep": True})


events = session.client("events")


def pause():
    """Stop the daily run (running batches still finish and are written)."""
    events.disable_rule(Name=OUTPUTS["DailyRuleName"])


def resume():
    events.enable_rule(Name=OUTPUTS["DailyRuleName"])


report(5)

## 10 · Remove it *(optional - deletes what this notebook created)*

Deletes the stack (Lambdas, roles, rules, layer). **The pipeline bucket is kept** (it holds batch files). Prompts and secrets are only deleted if you ask.

In [ ]:
CONFIRM_TEARDOWN = False
ALSO_DELETE_PROMPTS = False
ALSO_DELETE_SECRETS = False

if CONFIRM_TEARDOWN:
    cfn.delete_stack(StackName=STACK_NAME)
    cfn.get_waiter("stack_delete_complete").wait(StackName=STACK_NAME)
    print("stack deleted; bucket", CONFIG["pipeline_bucket"], "was kept")
    if ALSO_DELETE_PROMPTS:
        for ref in STATE.get("prompts", {}).values():
            agent.delete_prompt(promptIdentifier=ref["id"])
    if ALSO_DELETE_SECRETS:
        for name in (CONFIG["pg_secret_name"], CONFIG["snowflake_secret_name"]):
            sm.delete_secret(SecretId=name, RecoveryWindowInDays=7)
else:
    print("nothing deleted (set CONFIRM_TEARDOWN = True)")